# BÀI TẬP LỚN NHẬN DẠNG MẪU: HỆ THỐNG PHÁT HIỆN XÂM NHẬP MẠNG (NIDS)
## Phân loại Lưu lượng NetFlow v3 & Tối ưu hóa Rủi ro Tối thiểu Bayes (Bayes Minimum Risk)

---

### 1. Giới thiệu Bài toán và Bộ dữ liệu
- **Bài toán:** Phân loại và phát hiện các luồng mạng bất thường/tấn công độc hại (Intrusion Detection System) từ lưu lượng mạng NetFlow v3 viễn thông.
- **Bộ dữ liệu thực tế:** `NF-UNSW-NB15-v3` chứa hơn 2.36 triệu bản ghi NetFlow với 53 đặc trưng đo lường lưu lượng (throughput, packet inter-arrival time, TCP flags, packet lengths, TTL...).
- **Thách thức cốt lõi:**
  1. **Mất cân bằng lớp nghiêm trọng (Severe Class Imbalance):** Lưu lượng bình thường (Benign) chiếm ~94.6%, trong khi các cuộc tấn công tinh vi chỉ chiếm ~5.4%.
  2. **Tổn thất bất đối xứng (Asymmetric Misclassification Costs):** Chi phí thiệt hại khi bỏ lọt một cuộc tấn công ($C_{01}$) cao gấp hàng trăm lần chi phí điều tra một cảnh báo giả ($C_{10}$).

### 2. Thiết kế Mô hình So sánh
- **Phương án cơ sở (Baselines):**
  1. *Gaussian Naive Bayes (GNB):* Mô hình xác suất sinh giả định tính độc lập có điều kiện.
  2. *Logistic Regression (LR):* Mô hình phân biệt tuyến tính với ngưỡng chuẩn $\tau = 0.5$.
- **Phương án cải tiến (Improved Models):**
  1. *Decision Tree (CART):* Mô hình cây quyết định phi tuyến, tự động trích xuất luật phân tách.
  2. *Random Forest (RF):* Rừng cây tổ hợp (Bagging Ensemble) kết hợp **Lý thuyết quyết định Bayes (Bayes Decision Theory)** nhằm tối thiểu hóa rủi ro Bayes (Bayes Minimum Risk).

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries imported successfully!')

### 3. Tiền Xử Lý Dữ Liệu & Phân Tách Đặc Trưng
Loại bỏ các đặc trưng định danh topology mạng (`IPV4_SRC_ADDR`, `IPV4_DST_ADDR`, `FLOW_START_MILLISECONDS`, `DNS_QUERY_ID`) để đảm bảo tính tổng quát hóa.

In [ ]:
data_path = 'data/NF-UNSW-NB15-v3.csv'
parquet_path = 'processed/train_eval_data.parquet'

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
else:
    chunks_attack, chunks_benign = [], []
    for i, chunk in enumerate(pd.read_csv(data_path, chunksize=250000)):
        chunks_attack.append(chunk[chunk['Label'] == 1])
        chunks_benign.append(chunk[chunk['Label'] == 0].sample(frac=0.08, random_state=42 + i))
    df = pd.concat(chunks_attack + chunks_benign, ignore_index=True)

print(f'Total samples: {len(df):,}')
print('Class distribution:')
print(df['Attack'].value_counts())

### 4. Huấn Luyện & Đánh Giá 4 Mô Hình
Tiến hành chia tập Stratified Split (80/20) và chuẩn hóa dữ liệu cho GNB & Logistic Regression.

In [ ]:
non_feature_cols = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'DNS_QUERY_ID', 'Label', 'Attack']
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32)
y = df['Label'].to_numpy(dtype=np.int32)
attack_classes = df['Attack'].astype(str).to_numpy()

X_train, X_test, y_train, y_test, attack_train, attack_test = train_test_split(
    X, y, attack_classes, test_size=0.20, random_state=42, stratify=attack_classes
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Gaussian Naive Bayes': (GaussianNB(), X_train_scaled, X_test_scaled),
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), X_train_scaled, X_test_scaled),
    'Decision Tree (CART)': (DecisionTreeClassifier(max_depth=12, min_samples_leaf=10, random_state=42), X_train, X_test),
    'Random Forest (Ensemble)': (RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1), X_train, X_test)
}

results = {}
for name, (clf, xtr, xte) in models.items():
    t0 = time.time()
    clf.fit(xtr, y_train)
    fit_time = time.time() - t0
    y_prob = clf.predict_proba(xte)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'PR-AUC': average_precision_score(y_test, y_prob),
        'Train Time (s)': fit_time
    }

df_results = pd.DataFrame(results).T
print(df_results.to_string())

### 5. Lý Thuyết Quyết Định Bayes & Ma Trận Chi Phí Bất Đối Xứng
Trong thực tế an ninh mạng, một vụ xâm nhập bị bỏ sót ($C_{01}$) gây hậu quả nghiêm trọng hơn rất nhiều so với một cảnh báo giả ($C_{10}$).

Nguyên lý cực tiểu hóa rủi ro Bayes (Bayes Minimum Risk):
$$\mathcal{R}(\alpha | x) = \sum_{j} C(\alpha_i, \omega_j) P(\omega_j | x)$$

Luật quyết định tối ưu cho bài toán 2 lớp:
$$\text{Quyết định Attack (}\alpha_1\text{) nếu: } P(\text{Attack} | x) > \tau^* = \frac{C_{10} - C_{00}}{(C_{10} - C_{00}) + (C_{01} - C_{11})} \approx \frac{C_{10}}{C_{10} + C_{01}}$$

Với $C_{01} = 100$ và $C_{10} = 1$, ngưỡng quyết định tối ưu Bayes là:
$$\tau^* = \frac{1}{1 + 100} \approx 0.0099$$

In [ ]:
c10, c01 = 1.0, 100.0
prior_attack = 0.054
thresholds = np.linspace(0.001, 0.99, 100)

def calc_bayes_risk(y_true, y_prob, tau):
    y_p = (y_prob >= tau).astype(int)
    tp = np.sum((y_p == 1) & (y_true == 1))
    fn = np.sum((y_p == 0) & (y_true == 1))
    fp = np.sum((y_p == 1) & (y_true == 0))
    tn = np.sum((y_p == 0) & (y_true == 0))
    
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    risk = prior_attack * (fnr * c01) + (1.0 - prior_attack) * (fpr * c10)
    return risk

rf_clf = models['Random Forest (Ensemble)'][0]
rf_probs = rf_clf.predict_proba(X_test)[:, 1]
risks = [calc_bayes_risk(y_test, rf_probs, t) for t in thresholds]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, risks, color='#10B981', lw=2.5, label='Random Forest Expected Bayes Risk')
plt.axvline(x=c10/(c10+c01), color='crimson', linestyle='--', label=f'Theoretical Optimal Threshold tau* = {c10/(c10+c01):.4f}')
plt.axvline(x=0.5, color='gray', linestyle='-.', label='Standard Threshold tau = 0.50')
plt.xlabel('Decision Threshold (tau)')
plt.ylabel('Expected Bayes Risk')
plt.title('Bayes Risk Minimization under Asymmetric Misclassification Costs')
plt.legend()
plt.show()

### 6. Phân Tích Lỗi Chi Tiết (Error Analysis by Attack Class)
Đánh giá độ nhạy trên từng phân lớp tấn công cụ thể để phát hiện điểm mạnh và hạn chế của từng giải pháp.

In [ ]:
if os.path.exists('results/attack_class_error_analysis.csv'):
    df_err = pd.read_csv('results/attack_class_error_analysis.csv')
    print(df_err.to_string(index=False))
else:
    print('Please run final/train_models.py to export full error analysis!')

### 7. Kết Luận & Hạn Chế Của Giải Pháp
- **Kết luận:** Mô hình tổ hợp **Random Forest** kết hợp **Lý thuyết quyết định Bayes** đạt hiệu năng vượt trội toàn diện so với các Baseline (GNB, LR) trên cả độ chính xác, độ nhạy và rủi ro thiệt hại tối thiểu.
- **Hạn chế:** Một số loại tấn công dạng tàng hình (Backdoor / Fuzzers) có dấu vết lưu lượng gần giống phiên làm việc bình thường vẫn đòi hỏi phân tích chuỗi thời gian hoặc phân tích gói sâu (Deep Packet Inspection).